# 01 · Transformer From Scratch —— 与 GRU+Attention 同台拆机制

**家族位置**：`05_Transformer_NLP` 第 1 站。04 已把"记"从统计（HMM/CRF 手拉手）换成 RNN 传话、GRU 门控笔记本、最后 Bahdanau 聚光灯——本章把这盏"聚光灯"抽出来，**去掉循环**，拼成纯注意力的 Transformer，并在同一个 toy（S=6 复制/翻转）上与 GRU+Attention 同台，可视化注意力热力。

**学习目标**
1. 多头自注意力（MHA）拆解：Q/K/V 投影、缩放、softmax、加权求和、多头拼接
2. 因果 mask（下三角）+ 位置编码（sin/cos）为什么必须
3. GRU+Attention vs Transformer 同参数量级同台：复制/翻转 seq-acc
4. 注意力热力图：Transformer 学到的对角（复制）/反对角（翻转）对齐

## 1. 原理：从"带循环的聚光灯"到"全注意力"

### 通俗理解

**一句话**：04-03 的 Bahdanau 是"解码每一步单独回头看一眼"，Transformer 是"把所有位置互相看一遍"——没有循环、全部并行，一次前向就把 Q·Kᵀ 的 6×6 相似度矩阵算完。

**比喻**：GRU+Attention 像 1 个翻译官边听边翻（串行）；Transformer 像 6 个翻译官同时开会，每人手里拿着 Q（我要问什么）、K（我能答什么）、V（我的内容），互相打分（Q·Kᵀ）再取意见（加权 V）。**多头** = 6 个翻译官每人只关注一个侧面（语法/语义/位置），**残差+LayerNorm** = 每人发言后原稿保留（不被覆盖）。

### 结构账

```
MHA:   Q,K,V = xW_q, xW_k, xW_v   →  split 成 h 头  →  softmax(QKᵀ/√d_k)·V  →  concat → W_o
因果mask：解码自注意力只允许看 ≤ 当前位置（下三角 -inf），防未来泄露
位置编码：sin/cos(pos, 2i/2i+1)  加在 embedding 上（无循环 → 必须显式给"顺序"）
Encoder 层：x = LN(x + MHA(x,x,x)) → LN(x + FFN(x))     （Pre-LN 简化版）
Decoder 层：x = LN(x + causal-MHA(x)) → LN(x + cross-MHA(x, enc)) → LN(x + FFN(x))
```

- **与 04-03 同台**：同 S=6, vocab=8, emb16/hid32 量级，唯一变量是"循环+单步注意力" vs "全注意力并行"
- **评估**：seq-acc（整句全对）+ 注意力热力图（对角=复制，反对角=翻转）

In [ ]:
import sys
from pathlib import Path
import numpy as np
import torch
import torch.nn as nn
import matplotlib.pyplot as plt

ROOT = Path.cwd()
while ROOT != ROOT.parent and not (ROOT / "common").exists():
    ROOT = ROOT.parent
assert (ROOT / "common").exists(), "向上未找到 common 目录"
sys.path.insert(0, str(ROOT))

from common.data import make_copy_data, make_reverse_data, build_vocab
from common.models import Seq2SeqAttention, TransformerTiny, set_torch_seed
from common.utils import set_seed, setup_chinese_font, plot_attention, seq_accuracy, token_accuracy

set_seed(0)
setup_chinese_font()
FIGS = Path.cwd() / "figs"
FIGS.mkdir(exist_ok=True)
print("torch:", torch.__version__)

VOCAB, SEQ_LEN = 8, 6
BATCH, EPOCHS, LR = 32, 60, 8e-3
copy_X, copy_Y = make_copy_data(600, SEQ_LEN, VOCAB, seed=0)
rev_X, rev_Y = make_reverse_data(600, SEQ_LEN, VOCAB, seed=1)
print(f"复制 {len(copy_X)} 条 / 翻转 {len(rev_X)} 条 | 例: {copy_X[0]}→{copy_Y[0]} | 翻转例: {rev_X[0]}→{rev_Y[0]}")


## 2. 数据：同 04-03 的复制/翻转 toy（S=6, vocab=8）

复制学对角对齐，翻转学反对角（长程重排），600 条 60ep，CPU 秒级。

In [ ]:
def to_tensors(X, Y, seed=0):
    set_torch_seed(seed)
    xt = torch.tensor(X, dtype=torch.long)
    yt = torch.tensor(Y, dtype=torch.long)
    # teacher forcing：输入 [BOS, y0..y4]，目标 y0..y5
    bos = torch.full((yt.size(0), 1), VOCAB, dtype=torch.long)
    tgt_in = torch.cat([bos, yt[:, :-1]], dim=1)
    return xt, tgt_in, yt

def train_model(model, X, Y, epochs=EPOCHS, batch=BATCH, lr=LR, seed=0):
    xt, tgt_in, yt = to_tensors(X, Y, seed)
    opt = torch.optim.Adam(model.parameters(), lr=lr)
    lossf = nn.CrossEntropyLoss()
    hist = []
    n = xt.size(0)
    for ep in range(1, epochs + 1):
        model.train()
        perm = torch.randperm(n)
        tot = 0.0
        for i in range(0, n, batch):
            idx = perm[i:i+batch]
            logits, _ = model(xt[idx], tgt_in[idx]) if isinstance(model, Seq2SeqAttention) else (model(xt[idx], tgt_in[idx]), None)
            loss = lossf(logits.reshape(-1, VOCAB), yt[idx].reshape(-1))
            opt.zero_grad(); loss.backward(); opt.step()
            tot += loss.item() * len(idx)
        hist.append(tot / n)
    return hist

def eval_greedy(model, X, Y, max_len=SEQ_LEN, batch=64, seed=0):
    set_torch_seed(seed)
    model.eval()
    preds = []
    with torch.no_grad():
        for i in range(0, len(X), batch):
            xb = torch.tensor(X[i:i+batch], dtype=torch.long)
            p = model.greedy(xb, max_len)
            preds.extend(p.tolist())
    return preds, seq_accuracy(preds, Y), token_accuracy(preds, Y)

print("工具函数就绪")


## 3. 双模型同台：GRU+Attention vs Transformer（复制 & 翻转）

In [ ]:
results = {}
histories = {}
for task, (X, Y) in {"copy": (copy_X, copy_Y), "reverse": (rev_X, rev_Y)}.items():
    for name, maker in [("GRU+Attn", lambda: Seq2SeqAttention(VOCAB, 16, 32)),
                        ("Transformer", lambda: TransformerTiny(VOCAB, d_model=32, n_head=4, n_layer=2, d_ff=64))]:
        set_torch_seed(0)
        model = maker()
        h = train_model(model, X, Y, seed=0)
        preds, sacc, tacc = eval_greedy(model, X, Y, seed=0)
        n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
        results[(task, name)] = (sacc, tacc, n_params, model)
        histories[(task, name)] = h
        print(f"{task:8s} {name:12s} seq-acc={sacc:.3f} tok-acc={tacc:.3f} params={n_params}")

# fig1：loss 曲线 2×2
fig, axes = plt.subplots(2, 2, figsize=(11, 6.5), sharex=True)
for r, task in enumerate(["copy", "reverse"]):
    for name, color in [("GRU+Attn", "#4C72B0"), ("Transformer", "#DD8452")]:
        axes[r][0 if name=="GRU+Attn" else 1].plot(histories[(task, name)], color=color, label=name)
        axes[r][0 if name=="GRU+Attn" else 1].set_title(f"{task} · {name}", fontsize=10)
        axes[r][0 if name=="GRU+Attn" else 1].set_ylabel("CE loss")
for ax in axes[-1]: ax.set_xlabel("epoch")
plt.suptitle("训练曲线：GRU+Attention vs Transformer（同 toy 同预算）", fontsize=11)
plt.tight_layout()
plt.savefig(FIGS / "fig1_loss.png", dpi=150, bbox_inches="tight")
plt.show()

# fig2：seq-acc 柱状
fig, ax = plt.subplots(figsize=(6, 4))
tasks = ["copy", "reverse"]
gru = [results[(t, "GRU+Attn")][0] for t in tasks]
trf = [results[(t, "Transformer")][0] for t in tasks]
x = np.arange(2); w=0.34
b1 = ax.bar(x-w/2, gru, w, label="GRU+Attn", color="#4C72B0")
b2 = ax.bar(x+w/2, trf, w, label="Transformer", color="#DD8452")
for b in list(b1)+list(b2):
    ax.text(b.get_x()+b.get_width()/2, b.get_height()+0.015, f"{b.get_height():.3f}", ha="center", fontsize=9)
ax.set_xticks(x); ax.set_xticklabels(["复制 copy", "翻转 reverse"])
ax.set_ylim(0, 1.12); ax.set_ylabel("seq-acc（整句全对）")
ax.set_title("复制/翻转：两架构终局 seq-acc 同台")
ax.legend(); plt.tight_layout()
plt.savefig(FIGS / "fig2_bar.png", dpi=150, bbox_inches="tight")
plt.show()


## 4. 注意力热力：Transformer 学到的对齐

In [ ]:
# fig3：复制任务 Transformer 交叉注意力（单句，层2头平均）；fig4：翻转任务
idx = 0
src = torch.tensor([copy_X[idx]], dtype=torch.long)
trf_copy = results[("copy", "Transformer")][3]
trf_rev = results[("reverse", "Transformer")][3]

# 手动取 cross-attn：编码 → 逐层解码，抓最后一层 cross
with torch.no_grad():
    enc = trf_copy.encode(src)
    tgt_in = torch.cat([torch.full((1,1), VOCAB, dtype=torch.long), torch.tensor([copy_Y[idx][:-1]], dtype=torch.long)], dim=1)
    S = tgt_in.size(1)
    causal = torch.tril(torch.ones(S, S)).view(1,1,S,S)
    x = trf_copy.pos(trf_copy.emb(tgt_in))
    for li, lyr in enumerate(trf_copy.layers_dec):
        a,_ = lyr["self_attn"](x,x,x,mask=causal); x = lyr["ln1"](x+a)
        b, attn = lyr["cross_attn"](x, enc, enc, mask=None)
        if li == trf_copy.n_layer-1: cross = attn.mean(dim=1)[0].numpy()
        x = lyr["ln2"](x+b); x = lyr["ln3"](x+lyr["ffn"](x))

fig, ax = plt.subplots(figsize=(5.5, 4.6))
plot_attention(ax, cross, copy_X[idx], copy_Y[idx], title="Transformer 交叉注意力·复制（对角线=复制对齐）")
plt.tight_layout()
plt.savefig(FIGS / "fig3_attn_copy.png", dpi=150, bbox_inches="tight")
plt.show()

with torch.no_grad():
    enc = trf_rev.encode(torch.tensor([rev_X[idx]], dtype=torch.long))
    tgt_in = torch.cat([torch.full((1,1), VOCAB, dtype=torch.long), torch.tensor([rev_Y[idx][:-1]], dtype=torch.long)], dim=1)
    S = tgt_in.size(1)
    causal = torch.tril(torch.ones(S, S)).view(1,1,S,S)
    x = trf_rev.pos(trf_rev.emb(tgt_in))
    for li, lyr in enumerate(trf_rev.layers_dec):
        a,_ = lyr["self_attn"](x,x,x,mask=causal); x = lyr["ln1"](x+a)
        b, attn = lyr["cross_attn"](x, enc, enc, mask=None)
        if li == trf_rev.n_layer-1: cross = attn.mean(dim=1)[0].numpy()
        x = lyr["ln2"](x+b); x = lyr["ln3"](x+lyr["ffn"](x))

fig, ax = plt.subplots(figsize=(5.5, 4.6))
plot_attention(ax, cross, rev_X[idx], rev_Y[idx], title="Transformer 交叉注意力·翻转（反对角线=长程重排）")
plt.tight_layout()
plt.savefig(FIGS / "fig4_attn_reverse.png", dpi=150, bbox_inches="tight")
plt.show()


## 5. 总结与下一步

**本项目收获**

1. MHA/因果mask/位置编码全部手写，S=6 toy 上 Transformer 与 GRU+Attention 同台，seq-acc 双 1.0 量级
2. 注意力热力：复制=对角、翻转=反对角，"聚光灯"从 04-03 的逐步抬头变成一次性矩阵
3. 并行前向：无循环 → 训练时可全批并行，这是 08 家族 KV-Cache/FlashAttention 优化底座
4. Pre-LN 简化残差堆叠足以在小 toy 收敛，大模型还需 warmup/调度

**下一步**：`02_BERT_TextCls_FineTune`（Encoder-only）→ `03_GPT_LM_Mini`（Decoder-only）→ `04_T5_BART_Mini`（Encoder-Decoder）→ `05_Mamba_vs_Transformer_Mini`（拓展对照，与 04-04 呼应）。